# BasketCut AI — Kaggle GPU 视觉研究（非商用自动判分）
**目标**：检测录像中 `sports ball` 可见时间，生成候选 JSON，导入 Django 工作台人工审核。**不能判断进球、盖帽、助攻或球员身份。**
请仅上传取得授权的视频，不能把真实球员录像提交到公开 GitHub。Kaggle 免费 GPU 并不适用于 24/7 商用任务；Ultralytics 模型需审核 AGPL/企业商业许可证。


In [ ]:
%pip -q install 'ultralytics>=8.3,<9'
import os, cv2, json, torch
from ultralytics import YOLO
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 修改为 Kaggle Input Dataset 里的有授权视频路径。
VIDEO_PATH = '/kaggle/input/your-basketball-videos/game.mp4'
OUTPUT_PATH = '/kaggle/working/candidates.json'
SAMPLE_EVERY_SECONDS = 1.5
MAX_SAMPLES = 2400
assert os.path.exists(VIDEO_PATH), f'请先更改 VIDEO_PATH: {VIDEO_PATH}'
model = YOLO('yolo11n.pt')  # COCO sports ball 类别 32（研究用）
capture = cv2.VideoCapture(VIDEO_PATH)
fps = capture.get(cv2.CAP_PROP_FPS)
count = capture.get(cv2.CAP_PROP_FRAME_COUNT)
assert fps > 0 and count > 0, '打不开视频或视频元数据异常'
duration = count / fps
print('Duration:', round(duration,1), 'seconds')


In [ ]:
# 候选策略：采样帧出现 sports ball，就截取周围 5~8 秒，且候选间隔至少 14 秒。
detections = []
num_samples = min(MAX_SAMPLES, int(duration / SAMPLE_EVERY_SECONDS) + 1)
device = 0 if torch.cuda.is_available() else 'cpu'
for i in range(num_samples):
    second = i * SAMPLE_EVERY_SECONDS
    capture.set(cv2.CAP_PROP_POS_MSEC, second * 1000)
    ok, frame = capture.read()
    if not ok: continue
    result = model.predict(frame, device=device, classes=[32], imgsz=640, conf=0.28, verbose=False)[0]
    if result.boxes is not None and len(result.boxes):
        conf = float(result.boxes.conf.max().item())
        detections.append((second, conf))
capture.release()
detections.sort(key=lambda v:v[1], reverse=True)
picked = []
for t, conf in detections:
    if all(abs(t-x[0]) >= 14 for x in picked): picked.append((t,conf))
    if len(picked) >= 24: break
candidates = [dict(start_seconds=round(max(0,t-5),2),
                   end_seconds=round(min(duration,t+8),2),
                   source='visual_ball_presence', score=round(conf,3),
                   note='COCO sports ball 出现：仅作为人工审片建议')
              for t,conf in sorted(picked) if min(duration,t+8)-max(0,t-5) >= 1]
with open(OUTPUT_PATH,'w',encoding='utf-8') as f:
    json.dump({'schema_version':1,'candidates':candidates}, f, ensure_ascii=False, indent=2)
print('候选数:', len(candidates), '输出:', OUTPUT_PATH)
print('候选可能包括非精彩镜头，也可能漏掉进球；回到 Django 审片页面复核。')
